# 12: LUNA16 Ground-Truth Calibration (Step B)
3 strategies x 10 seeds x effnet_b0 on the 8,194-slice cohort (30 scans):
1. random_image (10 seeds) - Leaky baseline
2. block_shuffled (10 seeds) - Equal-cut proxy baseline
3. true_subject_split (10 seeds) - Genuine patient-level honest split

Computes the empirical calibration factor: how much the block proxy understates true patient leakage.


In [ ]:
import sys, os, json, time, types, glob, csv, shutil
import numpy as np
from collections import defaultdict, Counter
import torch
from PIL import Image

print("Mounted in /kaggle/input:", os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "none")

LIB = None
if os.path.isdir("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "splits.py" in files and "audit.py" in files and not LIB:
            LIB = root

if LIB:
    pkg = types.ModuleType("iqoth")
    pkg.__path__ = [LIB]
    sys.modules["iqoth"] = pkg
    sys.path.insert(0, LIB)
else:
    sys.path.insert(0, "src")

from iqoth import splits, external, engine, models, data
print("Imported iqoth successfully from", LIB)
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "GPU accelerator required!"


In [ ]:
LUNA_ROOT = None
if os.path.isdir("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "annotations.csv" in files and not LUNA_ROOT:
            LUNA_ROOT = root
assert LUNA_ROOT, "LUNA16 dataset not found in /kaggle/input"
print("Found LUNA16 root:", LUNA_ROOT)

nodule_uids = set()
ann_file = os.path.join(LUNA_ROOT, "annotations.csv")
if os.path.exists(ann_file):
    with open(ann_file) as f:
        for row in csv.DictReader(f):
            nodule_uids.add(row["seriesuid"])
print("Total nodule-positive UIDs in annotations.csv:", len(nodule_uids))

mhd_files = []
for sdir in ("subset0", "subset1"):
    p = os.path.join(LUNA_ROOT, sdir)
    if os.path.isdir(p):
        mhd_files.extend(glob.glob(p + "/**/*.mhd", recursive=True))
if not mhd_files:
    mhd_files = glob.glob(LUNA_ROOT + "/**/*.mhd", recursive=True)
print("Found scans in LUNA16:", len(mhd_files))

nodule_scans = [f for f in mhd_files if os.path.splitext(os.path.basename(f))[0] in nodule_uids]
normal_scans = [f for f in mhd_files if os.path.splitext(os.path.basename(f))[0] not in nodule_uids]
cohort = nodule_scans[:15] + normal_scans[:15]
print("Selected cohort scans:", len(cohort), f"({len(nodule_scans[:15])} nodule, {len(normal_scans[:15])} normal)")

# Slicing into /tmp/luna2d (NOT /kaggle/working to prevent output file pollution)
LUNA2D = "/tmp/luna2d"
os.makedirs(LUNA2D, exist_ok=True)

def read_mhd_raw(mhd_path):
    header = {}
    with open(mhd_path, "r", errors="ignore") as f:
        for line in f:
            if "=" in line:
                k, v = line.split("=", 1)
                header[k.strip()] = v.strip()
    raw_name = header.get("ElementDataFile")
    raw_path = os.path.join(os.path.dirname(mhd_path), raw_name)
    dims = [int(x) for x in header.get("DimSize", "512 512 0").split()]
    depth, height, width = dims[2], dims[1], dims[0]
    arr = np.fromfile(raw_path, dtype=np.int16).reshape((depth, height, width))
    return arr

t0 = time.time()
for mhd in cohort:
    suid = os.path.splitext(os.path.basename(mhd))[0]
    cls = "nodule_cases" if suid in nodule_uids else "normal_cases"
    out_dir = os.path.join(LUNA2D, cls, suid)
    if os.path.isdir(out_dir) and len(os.listdir(out_dir)) > 20:
        continue
    os.makedirs(out_dir, exist_ok=True)
    vol = read_mhd_raw(mhd)
    vol = np.clip(vol, -1000, 400)
    vol = ((vol + 1000) / 1400.0 * 255.0).astype(np.uint8)
    for idx, sl in enumerate(vol):
        im = Image.fromarray(sl).resize((128, 128))
        im.save(os.path.join(out_dir, f"slice_{idx:04d}.png"))
print(f"Slicing complete in {(time.time()-t0):.1f}s")

ext = external.adapter_folder_per_subject(LUNA2D)
print(f"Loaded {len(ext)} slice records across {len(set(r['subject_id'] for r in ext))} subjects")


In [ ]:
# Assign equal-cut proxy blocks (15 equal blocks per class, mimicking build_registry)
for cls in sorted(set(r["class_name"] for r in ext)):
    c_recs = [r for r in ext if r["class_name"] == cls]
    c_recs.sort(key=lambda r: r["pos"])
    blocks = np.array_split(np.arange(len(c_recs)), 15)
    for b_idx, idxs in enumerate(blocks):
        for i in idxs:
            c_recs[i]["block_id"] = f"{c_recs[0]['label']}_{b_idx}"

print("Proxy blocks assigned: 15 per class. Sample:")
for cls in sorted(set(r["class_name"] for r in ext)):
    c_recs = [r for r in ext if r["class_name"] == cls]
    print(f"  {cls}: {len(c_recs)} images, blocks={len(set(r['block_id'] for r in c_recs))}")


In [ ]:
SEEDS = [42, 123, 2024, 7, 2718, 1, 2, 3, 4, 5]
MODEL = "effnet_b0"
STRATEGIES = ["random_image", "block_shuffled", "true_subject_split"]

RESULTS = "/kaggle/working/results"
OUT = f"{RESULTS}/12_luna_calibration"
os.makedirs(OUT, exist_ok=True)

done = {os.path.basename(f).replace("_result.json", "")
        for f in os.listdir(OUT) if f.endswith("_result.json")} if os.path.isdir(OUT) else set()
print("Already complete:", len(done), "Total target runs:", len(SEEDS) * len(STRATEGIES))

t_start = time.time()
for strat in STRATEGIES:
    for seed in SEEDS:
        run_name = f"{MODEL}__{strat}__seed{seed}"
        if run_name in done:
            print("Skip:", run_name)
            continue
        
        print("\n>>> RUNNING:", run_name, f"(elapsed: {(time.time()-t_start)/60:.1f}m)")
        r_copy = [dict(r) for r in ext]
        
        if strat == "true_subject_split":
            external.true_subject_split(r_copy, seed=seed)
            for r in r_copy:
                r["strategy"] = "true_subject_split"
                r["split_seed"] = seed
                r["block_id"] = r["subject_id"]
            res = engine.run_experiment(LUNA2D, MODEL, strategy=None, seed=seed, out_dir=OUT,
                                        records=r_copy, verbose=False)
        else:
            res = engine.run_experiment(LUNA2D, MODEL, strategy=strat, seed=seed, out_dir=OUT,
                                        records=r_copy, verbose=False)
        
        print(f"    Finished {run_name}: acc={res['accuracy']:.2f}% (train_secs={res['train_seconds']}s)")


In [ ]:
import statistics as st
from scipy import stats

all_res = engine.collect_results(OUT)
by_strat = defaultdict(list)
for r in all_res:
    strat = r.get("strategy")
    if not strat:
        if "true_subject_split" in r["run_name"]:
            strat = "true_subject_split"
    by_strat[strat].append(r)

print("=" * 70)
print("LUNA16 CALIBRATION SUMMARY (3 strategies x 10 seeds effnet_b0)")
print("=" * 70)

summary = {}
for strat in STRATEGIES:
    runs = by_strat[strat]
    accs = [r["accuracy"] for r in runs]
    f1s = [r["macro_f1"] for r in runs]
    m_acc = float(st.mean(accs)) if accs else 0.0
    s_acc = float(st.stdev(accs)) if len(accs) > 1 else 0.0
    summary[strat] = {
        "n_seeds": len(accs),
        "mean_acc": round(m_acc, 2),
        "sd_acc": round(s_acc, 2),
        "mean_macro_f1": round(float(st.mean(f1s)), 2) if f1s else 0.0,
        "all_accs": accs
    }
    print(f"{strat:<20}: acc = {m_acc:6.2f}% +/- {s_acc:4.2f}%  (n={len(accs)})")

acc_rand = summary["random_image"]["mean_acc"]
acc_shuff = summary["block_shuffled"]["mean_acc"]
acc_true = summary["true_subject_split"]["mean_acc"]

delta_proxy = acc_rand - acc_shuff
delta_true = acc_rand - acc_true
ratio = delta_true / delta_proxy if abs(delta_proxy) > 1e-5 else 1.0

summary["delta_proxy_pp"] = round(delta_proxy, 2)
summary["delta_true_pp"] = round(delta_true, 2)
summary["correction_factor"] = round(ratio, 3)

iqoth_proxy_drop = 9.36
iqoth_calibrated_leakage = iqoth_proxy_drop * ratio
summary["iqoth_calibrated_leakage_pp"] = round(iqoth_calibrated_leakage, 2)

print("-" * 70)
print(f"Proxy Grouping Drop (delta_proxy):   {delta_proxy:6.2f} pp")
print(f"True Patient Leakage Drop (delta_true):{delta_true:6.2f} pp")
print(f"Correction Factor (delta_true / delta_proxy): {ratio:.3f}x")
print(f"Calibrated IQ-OTH True Leakage:       {iqoth_calibrated_leakage:6.2f} pp")

halt_inversion = acc_true > acc_shuff
summary["halt_inversion"] = halt_inversion
print("\nHALT CONDITION CHECK:")
if halt_inversion:
    print("  [HALT TRIGGERED]: true_subject_split scored ABOVE block_shuffled! Inverts argument.")
else:
    print(f"  [CLEARED]: true_subject_split ({acc_true:.2f}%) <= block_shuffled ({acc_shuff:.2f}%).")

with open(f"{OUT}/luna_calibration_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("\nSaved results to luna_calibration_summary.json")
